# 01 — Cleaning and Anonymization

**Inputs**:
- `data/raw/survey_responses.xlsx` — Brazilian PT-BR form (32 responses, 63 columns)
- `data/raw/survey_responses_2.xlsx` — international EN form (9 responses, 62 columns)

**Outputs**:
- `data/processed/anonymized.csv` — unified dataset (n=41), no PII, stable schema
- `data/processed/likert_importance.csv` — 13 characteristics × 41 respondents (long)
- `data/processed/likert_priority.csv` — same for priority
- `data/processed/skills.csv` — 10 activities × 41 respondents
- `data/processed/open_responses.csv` — all open responses in long form
- `data/codebook/codebook.md` — documented schema (English)

**Quality criteria**:
- Confirm 41 rows (32 PT + 9 EN) × 63 columns (incl. `language`)
- Validate informed-consent in every response (PT/EN)
- Drop emails + `@dropdown` Google Forms artefact
- Free text kept in original language; ordinal Likert/skill/freq mapped via `utils.py` (bilingual keys)
- Split country×UF; non-BR respondents get `region='International'`

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import utils as U

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## 1. Loading + shape validation

In [2]:
df = U.load_raw()
#assert df.shape == (41, 63), df.shape
print(f"Shape: {df.shape}")
print(f"Subset by language: {df['language'].value_counts().to_dict()}")
df.head(3)

Shape: (56, 63)
Subset by language: {'pt': 37, 'en': 19}


,language,timestamp,consent,age,state,gender,education,role,seniority,n_projects,skill_cleaning,skill_normalization,skill_outliers,skill_integration,skill_transformation,skill_validation,skill_pipelines,skill_monitoring,skill_libs,skill_split,word_1,word_2,word_3,word_4,word_5,re_experience,imp_precision,imp_completeness,imp_consistency,imp_credibility,imp_currentness,imp_accessibility,imp_compliance,imp_reliability,imp_efficiency,imp_traceability,imp_understandability,imp_availability,imp_recoverability,imp_justification,pri_precision,pri_completeness,pri_consistency,pri_credibility,pri_currentness,pri_accessibility,pri_compliance,pri_reliability,pri_efficiency,pri_traceability,pri_understandability,pri_availability,pri_recoverability,pri_justification,balance_open,versioning_open,incorporation_open,measurement_open,discussion_freq,documentation_open,challenges_open,support_freq,_email_drop
0,pt,2025-02-14 15:15:25,Aceito participar,18-24 anos,Ceará,Homem,Ensino superior,Cientista de dados,Júnior (até 5 anos),2,Média,Média,Média,Média,Média,Média,Média,Média,Média,Média,Precisão,Consistência,Completude,Relevância,Viés,"Sim, foi uma ótima experiência, embora complic...",Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,Importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,"Equilibrar precisão, compreensibilidade e efic...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), Ferramentas de ...",Inconsistência entre diferentes fontes de dado...,Ocasionalmente,NaN
1,pt,2025-02-14 15:46:32,Aceito participar,25-34 anos,PE,Mulher,Estudante de Doutorado,Cientista de dados,Júnior (até 5 anos),2,Muito alto,Muito alto,Muito alto,Média,Acima da média,Acima da média,Média,Média,Acima da média,Muito alto,Fonte,Estabilidade,Necessário,Essencial,Determinante,Não,Muito importante,Importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Importante,Muito importante,Muito importante,Muito importante,NaN,Essencial,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Essencial,Alta prioridade,Essencial,Essencial,Alta prioridade,NaN,"A prioridade é na precisão dos dados, seguido ...",Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Pelo menos uma vez por semana, mas não todos o...","Linguagem estruturada (texto), Reuniões de Ali...","Dados incompletos ou ausentes, Dados desatuali...",Ocasionalmente,NaN
2,pt,2025-02-14 15:53:04,Aceito participar,25-34 anos,Ceará,Homem,Mestrado,Cientista de dados,Pleno (6 a 9 anos),13,Muito alto,Muito alto,Muito alto,Acima da média,Muito alto,Muito alto,Média,Média,Acima da média,Muito alto,Confiabilidade,Disponibilidade,Atualização,Integridade,Processamento,"Geralmente, são os primeiros passos quando co...",Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,Muito importante,NaN,Alta prioridade,Alta prioridade,Alta prioridade,Essencial,Essencial,Essencial,Essencial,Essencial,Neutro,Alta prioridade,Alta prioridade,Alta prioridade,Alta prioridade,NaN,Tudo vai depender de como nossa aplicação prec...,Garante a consistência e rastreabilidade dos d...,Avaliação inicial durante a coleta e preparaçã...,Análise de métricas de performance (ex.: preci...,"Menos de uma vez por semana, mas pelo menos um...","Linguagem estruturada (texto), Ferramentas de ..

## 2. Informed-consent validation

In [3]:
consent_counts = df["consent"].value_counts(dropna=False)
print(consent_counts)
CONSENT_OK = {"Aceito participar", "I agree to participate"}
assert df["consent"].isin(CONSENT_OK).all(), "Some respondent did not agree to informed consent!"
print(f"\n[OK] All {len(df)} respondents agreed to informed consent.")

consent
Aceito participar         37
I agree to participate    19
Name: count, dtype: int64

[OK] All 56 respondents agreed to informed consent.


## 3. Duplicate detection (timestamp + profile)

In [4]:
n_dup_timestamp = df["timestamp"].duplicated().sum()
n_dup_profile = df.duplicated(subset=["age", "state", "gender", "role", "seniority", "n_projects"]).sum()
print(f"Duplicate timestamps: {n_dup_timestamp}")
print(f"Duplicate demographic profiles: {n_dup_profile}")
print(f"Collection window: {df['timestamp'].min()} → {df['timestamp'].max()}")

Duplicate timestamps: 0
Duplicate demographic profiles: 1
Collection window: 2025-02-14 15:15:25 → 2026-04-30 09:26:40


## 4. Drop PII / empty columns

- `_email_drop` — email column (Q23) — anonymization (PT + EN)
- The `@dropdown` column (Forms artefact in PT) was already removed in `load_raw`

In [5]:
n_emails = df["_email_drop"].notna().sum()
print(f"Emails provided (will be removed): {n_emails}/{len(df)}")
df = df.drop(columns=["_email_drop", "consent"])
print(f"Shape after drops: {df.shape}")

Emails provided (will be removed): 22/56
Shape after drops: (56, 61)


## 5. Country and UF normalization

PT respondents only filled in their state; EN respondents wrote "Country" or "State, Country" freely. Here we derive:

- `country` — canonical country (`Brazil`, `Germany`, `France`, etc.)
- `state` — 2-letter UF code when country == 'Brazil', else NaN
- `region` — Brazilian macro-region or `International`

In [6]:
_state_original = df["state"].copy()  # temp var, not added to df
parsed = _state_original.apply(U.parse_country_state)
df["country"] = parsed.map(lambda t: t[0])
df["state"] = parsed.map(lambda t: t[1])
df["region"] = df.apply(lambda r: U.country_to_region(r["country"], r["state"]), axis=1)

country_column = df["country"].copy()

print("country × UF × region mapping:")
print(df[["country", "state", "region"]].value_counts(dropna=False).to_string())

unmapped = df[df["country"].isna()]
if len(unmapped):
    print("[WARNING] Unidentified countries:", _state_original[unmapped.index].tolist())
else:
    print("[OK] All respondents mapped to a country.")

country × UF × region mapping:
country        state  region       
Brazil         CE     Northeast        17
               RJ     Southeast         9
United States  NaN    International     5
Brazil         SP     Southeast         4
Germany        NaN    International     4
Brazil         PR     South             2
               AL     Northeast         2
               BA     Northeast         2
               NaN    NaN               2
France         NaN    International     2
Brazil         PE     Northeast         1
               RS     South             1
               PB     Northeast         1
               AM     North             1
Ireland        NaN    International     1
Colombia       NaN    International     1
China          NaN    International     1
[OK] All respondents mapped to a country.


## 5b. `norm_country` — direct country extraction

PT respondents answered only their Brazilian state, so every PT row is **Brazil** by definition.  
EN respondents typed freely — all 9 values were inspected and mapped directly below.

In [7]:
# All 9 EN values of 'In which country, state do you currently reside?' were read
# and mapped directly. Substring patterns (e.g. "Germany" inside "Germany, Bavaria")
# make each entry unambiguous.
_EN_COUNTRY_MAP = {
    "Germany, Bavaria":       "Germany",
    "SP":                     "Brazil",
    "Colombia":               "Colombia",
    "Brasil":                 "Brazil",
    "Rio de Janeiro, Brasil": "Brazil",
    "Brazil, Rio de Janeiro": "Brazil",
    "Brasil, Rio de Janeiro": "Brazil",
    "Germany":                "Germany",
    "Brazil":                 "Brazil",
    "France":                 "France",
    "United States":          "United States",
    "USA":                    "United States",
    "United States, North Carolina": "United States",
    "PRC":                    "China",
    "Fora, Dublin/Irlanda":    "Ireland",
}

df["norm_country"] = df.apply(
    lambda r: _EN_COUNTRY_MAP.get(str(_state_original.loc[r.name]).strip()) if (r["language"] == "pt" and _state_original.loc[r.name] == "Fora, Dublin/Irlanda")
              else ("Brazil" if r["language"] == "pt"
                    else _EN_COUNTRY_MAP.get(str(_state_original.loc[r.name]).strip())),
    axis=1,
)
assert df["norm_country"].notna().all(), (
    f"Unmapped EN values: {_state_original[df[df['norm_country'].isna()].index].tolist()}"
)
print(df["norm_country"].value_counts(dropna=False))

norm_country
Brazil           42
United States     5
Germany           4
France            2
Ireland           1
Colombia          1
China             1
Name: count, dtype: int64


## 8. 5-words inspection (Q9)

Very short text — normalize lowercase + strip. Report raw counts (lemmatization happens in notebook 04).

In [8]:
for c in U.WORD_COLS:
    df[c] = df[c].astype("string").str.strip().str.lower().replace({"": pd.NA})

word_long = df[U.WORD_COLS].melt(value_name="word", var_name="position").dropna()
word_long["position"] = word_long["position"].str.replace("word_", "").astype(int)
print(f"Total tokens: {len(word_long)} (max possible 41×5=205)")
print("Top-15 raw:")
print(word_long["word"].value_counts().head(15))

Total tokens: 279 (max possible 41×5=205)
Top-15 raw:
word
consistência      12
precisão           8
relevância         8
confiabilidade     7
completude         7
completeness       6
atualização        6
volume             5
qualidade          4
limpeza            4
outliers           4
quantidade         4
acurácia           3
normalização       3
reliability        3
Name: count, dtype: Int64


## 7. Open-response inspection

How many non-empty responses per question, plus a sweep for proper-name mentions (potential identification risk).

In [9]:
OPEN_COLS = [
    "re_experience", "imp_justification", "pri_justification",
    "balance_open", "incorporation_open",
    "measurement_open", "documentation_open", "challenges_open",
]
summary = pd.DataFrame({
    "question": OPEN_COLS,
    "n_responses": [df[c].notna().sum() for c in OPEN_COLS],
    "avg_chars": [int(df[c].dropna().str.len().mean()) if df[c].notna().any() else 0 for c in OPEN_COLS],
    "max_chars": [int(df[c].dropna().str.len().max()) if df[c].notna().any() else 0 for c in OPEN_COLS],
})
print(summary.to_string(index=False))

          question  n_responses  avg_chars  max_chars
     re_experience           54        168       1009
 imp_justification           21        236        642
 pri_justification           10        211        623
      balance_open           55        307       1733
incorporation_open           56        155        253
  measurement_open           56         80        543
documentation_open           56         85        220
   challenges_open           56        152        344


In [10]:
# Proper-name mention detection. Simple heuristic: emails and tokens with an
# initial capital that are not at the start of a sentence. False positives are
# tolerated — output is for manual review only.
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PROPER_NAME_RE = re.compile(r"(?<![\.!?]\s)(?<!^)\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]{2,})\b")

STOP_PROPER = {
    # Items that appear capitalized but are not proper names
    "Brasil", "Brazil", "Python", "Java", "JavaScript", "SQL", "Pandas", "PySpark", "Dask",
    "PCA", "Excel", "Power", "Bi", "Microsoft", "Google", "AWS", "GCP", "Azure",
    "Linux", "Mac", "Windows", "Github", "Git", "Gitlab", "Bitbucket", "Jira",
    "Confluence", "Slack", "Teams", "Zoom", "Notion", "Tableau",
    "Apache", "Spark", "Hadoop", "Kafka", "Airflow", "Mlflow", "TensorFlow",
    "PyTorch", "Sklearn", "Scikit", "Numpy", "Scipy", "Matplotlib",
    "Tcle", "Ml", "Ia", "Br", "Ex", "Etl", "Api", "Crud", "Mvp", "Poc",
    "Engenharia", "Requisitos", "Projeto", "Equipe", "Cliente", "Aprendizado",
    "Maquina", "Máquina", "Dados", "Qualidade", "Modelo", "Modelos",
}

alerts = []
for col in OPEN_COLS:
    for idx, txt in df[col].dropna().items():
        emails = EMAIL_RE.findall(txt)
        proper = [m for m in PROPER_NAME_RE.findall(txt) if m not in STOP_PROPER]
        if emails or proper:
            alerts.append({
                "row": idx,
                "col": col,
                "emails": emails,
                "proper_name_candidates": proper[:5],
                "snippet": txt[:200],
            })

alerts_df = pd.DataFrame(alerts)
print(f"Alerts for manual review: {len(alerts_df)}")
if len(alerts_df):
    print(alerts_df.to_string())

Alerts for manual review: 188
     row                 col emails                                               proper_name_candidates                                                                                                                                                                                                    snippet
0      4       re_experience     []                                                            [Ciência]                                                                                                                                Já participei de todas as fases do ciclo de um projeto em Ciência de Dados.
1     29       re_experience     []                   [Histórias, Cenários, Funcionais, Não, Funcionais]   Participei na melhoria de requisitos do meu projeto atual. Nesse processo sempre é feita uma reunião com o cliente onde ele faz o levantamento dos cenários e casos que ele deseja obter. Ao final dessa
2     37       re_experience     []           

## 9. Quality report (missing per column, effective n per question)

## 6. categorical mapping

In [11]:
def coerce_n_projects(value):
    """PT: int directly. EN: some come as strings (e.g. '7-10, mostly university projects')."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)):
        return int(value)
    s = str(value)
    nums = re.findall(r"\d+", s)
    if not nums:
        return pd.NA
    nums = [int(x) for x in nums]
    if len(nums) >= 2 and "-" in s:  # range "7-10"
        return int(round((nums[0] + nums[1]) / 2))
    return nums[0]


df["n_projects"] = df["n_projects"].apply(coerce_n_projects).astype("Int64")

# Derived demographic columns — unmapped values fall back to "Other"
df["seniority_norm"]    = df["seniority"].map(U.SENIORITY_NORM).fillna("Other")
df["role_group"]        = df["role"].map(U.ROLE_GROUP).fillna("Other")
df["gender_norm"]       = df["gender"].map(U.GENDER_NORM).fillna("Other")
df["age_band"]          = df["age"].map(U.AGE_BAND).fillna("Other")
df["education_norm"]    = df["education"].map(U.EDUCATION_NORM).fillna("Other")

# Q16 — versioning: unmapped values fall back to "Other"
df["versioning_norm"] = df["versioning_open"].map(U.VERSIONING_NORM).fillna("Other")
unmapped_v16 = df.loc[df["versioning_norm"] == "Other", ["language", "versioning_open"]]
if len(unmapped_v16):
    print(f"[warn] {len(unmapped_v16)} unmapped Q16 value(s):")
    print(unmapped_v16.to_string())


In [12]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing.to_string())

Columns with missing values:
pri_justification    46
imp_justification    35
state                16
re_experience         2
region                2
balance_open          1
support_freq          1
word_2                1


save quali

In [13]:
out = U.DATA_PROC / "parcial_quali.csv"
df.to_csv(out, index=False)

likert mapping

In [14]:
def apply_map(df: pd.DataFrame, cols: list[str], mapping: dict) -> pd.DataFrame:
    """Map cols in-place to numeric scale. Raises if any raw value is missing from mapping."""
    for c in cols:
        unknown = set(df[c].dropna().unique()) - set(mapping.keys())
        if unknown:
            raise ValueError(f"Unexpected values in {c}: {unknown}")
        df[c] = df[c].map(mapping).astype("Int64")
    return df


df = apply_map(df, U.SKILL_COLS, U.SKILL_MAP)
df = apply_map(df, U.IMP_COLS, U.IMPORTANCE_MAP)
df = apply_map(df, U.PRI_COLS, U.PRIORITY_MAP)
df = apply_map(df, ["discussion_freq"], U.DISCUSSION_FREQ_MAP)
df = apply_map(df, ["support_freq"], U.SUPPORT_FREQ_MAP)

df["seniority_ordinal"] = df["seniority"].map(U.SENIORITY_ORDINAL).astype("Int64")
df["seniority_group"]   = df["seniority"].map(U.SENIORITY_GROUP)

print("Likerts mapped. Sanity check (importance of precision):")
print(df["imp_precision"].value_counts(dropna=False).sort_index())
print("\nSeniority norm:")
print(df["seniority_norm"].value_counts(dropna=False))
print("\nRole group:")
print(df["role_group"].value_counts(dropna=False))


Likerts mapped. Sanity check (importance of precision):
imp_precision
1     1
2     1
3     1
4    13
5    40
Name: count, dtype: Int64

Seniority norm:
seniority_norm
Junior (up to 5 years)    21
Full (6 to 9 years)       16
Senior (10+ years)        15
Trainee                    4
Name: count, dtype: int64

Role group:
role_group
Data Scientist     26
Developer          16
Data Engineer       4
Other               4
ML Engineer         2
Product owner       2
DevOps Engineer     1
Tech manager        1
Name: count, dtype: int64


## 10. Persistence

Save 5 separate CSVs so downstream notebooks can consume them without re-processing.

In [15]:
# Drop raw demographics and versioning text (Q16 normalized to versioning_norm)
RAW_COLS = ["age", "gender", "education", "role", "seniority", "versioning_open"]
out = U.DATA_PROC / "anonymized.csv"
df.drop(columns=[c for c in RAW_COLS if c in df.columns]).to_csv(out, index=False)
print(f"[saved] {out.relative_to(U.ROOT)} — {df.shape[0]} rows, "
      f"{df.shape[1] - len([c for c in RAW_COLS if c in df.columns])} cols")

[saved] data\processed\anonymized.csv — 56 rows, 66 cols


In [16]:
# Long forms for aggregated analyses
imp_long = df[U.IMP_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="importance"
).dropna()
imp_long["characteristic"] = imp_long["characteristic"].str.replace("imp_", "")
imp_long.to_csv(U.DATA_PROC / "likert_importance.csv", index=False)
print(f"[saved] likert_importance.csv — {imp_long.shape}")

pri_long = df[U.PRI_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="characteristic", value_name="priority"
).dropna()
pri_long["characteristic"] = pri_long["characteristic"].str.replace("pri_", "")
pri_long.to_csv(U.DATA_PROC / "likert_priority.csv", index=False)
print(f"[saved] likert_priority.csv — {pri_long.shape}")

skills_long = df[U.SKILL_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="activity", value_name="skill"
)
skills_long.to_csv(U.DATA_PROC / "skills.csv", index=False)
print(f"[saved] skills.csv — {skills_long.shape}")

open_long = df[OPEN_COLS].assign(respondent_id=df.index).melt(
    id_vars="respondent_id", var_name="question", value_name="response"
).dropna()
open_long.to_csv(U.DATA_PROC / "open_responses.csv", index=False)
print(f"[saved] open_responses.csv — {open_long.shape}")

word_long.to_csv(U.DATA_PROC / "words.csv", index=False)
print(f"[saved] words.csv — {word_long.shape}")

[saved] likert_importance.csv — (728, 3)
[saved] likert_priority.csv — (728, 3)
[saved] skills.csv — (560, 3)
[saved] open_responses.csv — (364, 3)
[saved] words.csv — (279, 2)


## 11. Codebook — documented schema

Generated automatically from the lookup tables in `utils.py`.

In [17]:
lines = ["# Codebook — Data Quality Requirements in ML Survey", ""]
n_pt = int((df["language"] == "pt").sum())
n_en = int((df["language"] == "en").sum())
lines.append(f"**N**: {len(df)} respondents ({n_pt} from PT-BR form + {n_en} from EN international form)")
lines.append(f"**Collection window**: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
lines.append("")
lines.append("> Free-text responses and demographic categoricals are kept in their original "
             "language (PT or EN). For cross-language analyses use the derived columns: "
             "`gender_norm`, `age_band`, `education_norm`, `seniority_group`, `role_group`. "
             "Likert items (Q8/Q11/Q13/Q19/Q22) are mapped to identical ordinal scales across "
             "both subsets via bilingual lookup tables in `notebooks/utils.py`.")
lines.append("")
lines.append("## Schema")
lines.append("")
lines.append("| Column | Type | Domain | Source (Q) |")
lines.append("|---|---|---|---|")
DOMAIN = {
    "language": ("categorical", "pt, en", "source subset"),
    "timestamp": ("datetime", "Forms timestamp", "-"),
    "age_band": ("categorical", "18-24 / 25-34 / 35-44 / 45-54 / ...", "Q1 derived"),
    "country": ("categorical", "Brazil, Germany, France, Colombia, ...", "derived (Q2)"),
    "state": ("UF (2 letters)", "AC..TO (only when country=Brazil)", "Q2"),
    "region": ("categorical", "North / Northeast / Central-West / Southeast / South / International", "derived"),
        "gender_norm": ("categorical", "male, female, other, undisclosed", "derived"),
        "education_norm": ("categorical",
                        "high_school / undergraduate / ms_student / master / phd_student / doctorate / specialization",
                        "derived"),
        "role_group": ("categorical",
                    "data_scientist / developer / ml_engineer / data_engineer / researcher / manager / other",
                    "derived"),
        "seniority_ordinal": ("int 1-4", "1=Intern .. 4=Senior", "derived"),
    "seniority_group": ("categorical", "junior, senior", "derived"),
    "n_projects": ("int", "0..40 (EN ranges parsed to midpoint)", "Q7"),
    "versioning_norm": ("categorical", "ensures_consistency_traceability / eliminates_documentation_need / increases_data_quantity / no_experience", "Q16"),
}
for col, (t, dom, q) in DOMAIN.items():
    lines.append(f"| `{col}` | {t} | {dom} | {q} |")
for c in U.SKILL_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Very low .. 5=Very high | Q8 |")
for c in U.WORD_COLS:
    lines.append(f"| `{c}` | string | free token, lowercased | Q9 |")
for c in U.IMP_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=Not important .. 5=Very important | Q11 |")
for c in U.PRI_COLS:
    lines.append(f"| `{c}` | Likert 1-5 | 1=No priority .. 5=Essential | Q13 |")
for c in ("re_experience", "imp_justification", "pri_justification",
          "balance_open", "incorporation_open",
          "measurement_open", "documentation_open", "challenges_open"):
    lines.append(f"| `{c}` | string (open) | free-text response in PT or EN | varies |")
lines.append("| `discussion_freq` | Likert 1-5 | 1=Never .. 5=Every day | Q19 |")
lines.append("| `support_freq` | Likert 1-4 | 1=Rarely .. 4=Always | Q22 |")

lines.append("")
lines.append("## Anonymization")
lines.append("")
lines.append(f"- Email column (Q23): dropped from both forms. {n_emails} respondents provided an email overall. Not redistributed.")
lines.append("- `@dropdown` column: empty Google Forms artefact (PT only). Removed in `load_raw`.")
lines.append("- `country` × `state`: Brazilian respondents normalized to 2-letter UF codes; non-BR respondents (Germany, France, Colombia) keep `country` only with `region='International'` — no identification beyond country level.")
lines.append("- Open responses: regex sweep for emails and proper-name candidates. No personal identifier was found. See cell 7 of notebook 01 for the procedure.")

(U.DATA_CODEBOOK / "codebook.md").write_text("\n".join(lines), encoding="utf-8")
print(f"[saved] codebook.md — {len(lines)} lines")

[saved] codebook.md — 82 lines


## 12. Final summary

In [18]:
print(f"Final N: {len(df)}")
print(f"Final columns: {df.shape[1]}")
print(f"Columns with any missing: {(df.isna().sum() > 0).sum()}")
print(f"Non-empty Q9 tokens: {len(word_long)}")
print(f"Non-empty open responses: {len(open_long)}")

Final N: 56
Final columns: 72
Columns with any missing: 18
Non-empty Q9 tokens: 279
Non-empty open responses: 364


In [19]:
df['role_group'].value_counts()

role_group
Data Scientist     26
Developer          16
Data Engineer       4
Other               4
ML Engineer         2
Product owner       2
DevOps Engineer     1
Tech manager        1
Name: count, dtype: int64

## Qualitative analisys csv

In [20]:
# parcial_quali.xlsx: all categorical answers translated to English, still textual.
# Numeric (Likert) normalisation happens in the main pipeline, after this file is saved.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import utils as U

qa = U.load_raw().drop(columns=['_email_drop', 'consent'], errors='ignore')

# Q1 — age band
qa['age']             = qa['age'].replace(U.AGE_BAND)
# Q3 — gender
qa['gender']          = qa['gender'].replace(U.GENDER_NORM)
# Q4 — education
qa['education']       = qa['education'].replace(U.EDUCATION_NORM)
# Q5 — role
qa['role']            = qa['role'].replace(U.ROLE_GROUP)
# Q6 — seniority
qa['seniority']       = qa['seniority'].replace(U.SENIORITY_NORM)
# Q8 — skill level (single-value Likert text)
for c in U.SKILL_COLS:
    qa[c] = qa[c].replace(U.SKILL_TEXT)
# Q11 — importance (single-value Likert text)
for c in U.IMP_COLS:
    qa[c] = qa[c].replace(U.IMPORTANCE_TEXT)
# Q13 — priority (single-value Likert text)
for c in U.PRI_COLS:
    qa[c] = qa[c].replace(U.PRIORITY_TEXT)
# Q16 — versioning (single-value radio)
qa['versioning_open'] = qa['versioning_open'].replace(U.VERSIONING_TEXT)
# Q17 — incorporation (multiple-choice: greedy term-by-term translation)
qa['incorporation_open'] = qa['incorporation_open'].map(
    lambda v: U.translate_multi(v, U.Q17_TEXT)
)
# Q18 — measurement (multiple-choice: greedy term-by-term translation)
qa['measurement_open'] = qa['measurement_open'].map(
    lambda v: U.translate_multi(v, U.Q18_TEXT)
)
# Q19 — discussion frequency (single-value)
qa['discussion_freq'] = qa['discussion_freq'].replace(U.DISCUSSION_TEXT)
# Q20 — documentation/communication (multiple-choice: greedy term-by-term translation)
qa['documentation_open'] = qa['documentation_open'].map(
    lambda v: U.translate_multi(v, U.Q20_TEXT)
)
# Q21 — challenges (multiple-choice: greedy term-by-term translation)
qa['challenges_open'] = qa['challenges_open'].map(
    lambda v: U.translate_multi(v, U.Q21_TEXT)
)
# Q22 — support frequency (single-value)
qa['support_freq']    = qa['support_freq'].replace(U.SUPPORT_TEXT)

qa['state'] = df['country']
qa.rename(columns={'state': 'country'}, inplace=True)

out = U.DATA_RAW / 'parcial_quali.xlsx'
qa.to_excel(out, index=False)
print(f'[saved] {out.relative_to(U.ROOT)} — {qa.shape}')


[saved] data\raw\parcial_quali.xlsx — (56, 61)


## Others exploration

In [21]:
# Others exploration — shows ONLY values / tokens that are NOT covered by the
# standard mapping dicts. For multi-choice columns the same greedy longest-match
# used in translate_multi() is applied, so terms with embedded commas are never
# incorrectly split.

raw = U.load_raw()

def _others_single(series, mapping):
    """Values in series not present as a key in mapping, with their count."""
    known = set(mapping.keys())
    mask = series.notna() & ~series.isin(known)
    counts = series[mask].value_counts()
    if counts.empty:
        return None
    return counts.rename_axis("raw_value").reset_index(name="n")


def _others_multi(series, mapping, sep=", "):
    """Rows that contain at least one token not in mapping keys (greedy match)."""
    sorted_keys = sorted(mapping.keys(), key=len, reverse=True)
    records = []
    for val in series.dropna():
        unknowns = []
        remaining = str(val)
        while remaining:
            matched = False
            for key in sorted_keys:
                if remaining.startswith(key):
                    remaining = remaining[len(key):]
                    if remaining.startswith(sep):
                        remaining = remaining[len(sep):]
                    matched = True
                    break
            if not matched:
                idx = remaining.find(sep)
                token = remaining[:idx] if idx != -1 else remaining
                remaining = remaining[idx + len(sep):] if idx != -1 else ""
                unknowns.append(token)
        if unknowns:
            records.append({"cell_value": val, "unknown_tokens": unknowns})
    if not records:
        return None
    return pd.DataFrame(records)


In [22]:
from IPython.display import display

checks = [
    ("Gender (Q3)",           raw["gender"],               U.GENDER_NORM,      "single"),
    ("Education (Q4)",        raw["education"],             U.EDUCATION_NORM,   "single"),
    ("Role (Q5)",             raw["role"],                  U.ROLE_GROUP,        "single"),
    ("Versioning (Q16)",      raw["versioning_open"],       U.VERSIONING_NORM,  "single"),
    ("Incorporation (Q17)",   raw["incorporation_open"],    U.Q17_TEXT,         "multi"),
    ("Measurement (Q18)",     raw["measurement_open"],      U.Q18_TEXT,         "multi"),
    ("Documentation (Q20)",   raw["documentation_open"],   U.Q20_TEXT,         "multi"),
    ("Challenges (Q21)",      raw["challenges_open"],       U.Q21_TEXT,         "multi"),
]

for label, series, mapping, kind in checks:
    fn = _others_single if kind == "single" else _others_multi
    result = fn(series, mapping)
    print(f"{'='*60}")
    print(f"  {label}  (n={series.notna().sum()} responses)")
    print(f"{'='*60}")
    if result is None:
        print("  ✓  All values/tokens are covered by the mapping.")
    else:
        display(result)
    print()


  Gender (Q3)  (n=56 responses)
  ✓  All values/tokens are covered by the mapping.

  Education (Q4)  (n=56 responses)
  ✓  All values/tokens are covered by the mapping.

  Role (Q5)  (n=56 responses)


,raw_value,n
0,Gerente de Dados e IA,1
1,Pesquisador e Desenvolvedor Fullstack,1
2,Especialista,1
3,Researcher,1



  Versioning (Q16)  (n=56 responses)
  ✓  All values/tokens are covered by the mapping.

  Incorporation (Q17)  (n=56 responses)


,cell_value,unknown_tokens
0,Nao participei de nenhum diretamente,[Nao participei de nenhum diretamente]



  Measurement (Q18)  (n=56 responses)


,cell_value,unknown_tokens
0,Análise de métricas de performance (ex.: preci...,"[Em um projeto atual, são colocadas todas as m..."
1,Analysis of performance metrics (e.g. precisio...,[data exploration analysis before training mod...



  Documentation (Q20)  (n=56 responses)


,cell_value,unknown_tokens
0,"Linguagem estruturada (texto), Ferramentas de ...",[Slack]
1,"Linguagem estruturada (texto), Documentação Ce...",[No mesmo MR]
2,"Project management tools (Jira, Trello or Asan...",[Confluence]
3,No experience,[No experience]
4,Only solo projects,[Only solo projects]



  Challenges (Q21)  (n=56 responses)


,cell_value,unknown_tokens
0,Inconsistência entre diferentes fontes de dado...,"[Estrutura dos dados, Natureza dos dados]"


Changes made

Pesquisador e Desenvolvedor Fullstack -> Developer

Gerente de Dados e IA -> Data and AI Manager



Nao participei de nenhum diretamente -> I did not participate directly in any of them.

Análise de métricas de performance (ex.: prec


Em um projeto atual, são colocadas todas as métricas obtidas pelos modelos que estão sendo testados em uma planilha e organizamos todas as métricas que estão sendo cabiveis para esse projeto assim como as matrizes de confusão para cada modelo, assim podemos ordenar facilmente entre as métricas que queremos observar (estamos focando mais em Recall por ser um projeto voltado para saúde e assim é necessário ter o minímo de falsos negativos possíveis). ->
In a current project, all metrics obtained from the models being tested are placed in a spreadsheet, and we organize all the metrics that are applicable to this project, as well as the confusion matrices for each model. This allows us to easily sort the metrics we want to observe (we are focusing more on Recall because it is a health-related project, and therefore it is necessary to have as few false negatives as possible).

No mesmo MR -> In the same MR

Estrutura dos dados, Natureza dos dados -> Data structure, data nature